In [6]:
import pandas as pd 
master_df = pd.read_csv("../data/processed/master_dataset.csv", low_memory=False)

In [7]:
master_df.columns

Index(['Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'address1',
       ...
       'Other Operating Expenses', 'Average Dilution Earnings',
       'Selling And Marketing Expense', 'Earnings From Equity Interest',
       'Gain On Sale Of Ppe', 'Write Off', 'Loss Adjustment Expense',
       'Net Policyholder Benefits And Claims', 'Policyholder Benefits Gross',
       'Policyholder Benefits Ceded'],
      dtype='object', length=258)

In [8]:
print(master_df.columns.tolist())

['Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'address1', 'city', 'state', 'zip', 'country', 'phone', 'website', 'industry', 'industryKey', 'industryDisp', 'sector', 'sectorKey', 'sectorDisp', 'longBusinessSummary', 'fullTimeEmployees', 'companyOfficers', 'auditRisk', 'boardRisk', 'compensationRisk', 'shareHolderRightsRisk', 'overallRisk', 'governanceEpochDate', 'compensationAsOfEpochDate', 'irWebsite', 'executiveTeam', 'maxAge', 'priceHint', 'previousClose', 'open', 'dayLow', 'dayHigh', 'regularMarketPreviousClose', 'regularMarketOpen', 'regularMarketDayLow', 'regularMarketDayHigh', 'dividendRate', 'dividendYield', 'exDividendDate', 'payoutRatio', 'fiveYearAvgDividendYield', 'beta', 'trailingPE', 'forwardPE', 'volume', 'regularMarketVolume', 'averageVolume', 'averageVolume10days', 'averageDailyVolume10Day', 'bid', 'ask', 'bidSize', 'askSize', 'marketCap', 'nonDilutedMarketCap', 'fiftyTwoWeekLow', 'fiftyTwoWeekHigh', 'allTimeHigh', 'allTimeLo

In [9]:
master_df.groupby('Ticker').size()
missing_counts = master_df.isna().sum()
missing_counts[missing_counts > 0].sort_values(ascending=False)

Earnings From Equity Interest           12560
Salaries And Wages                      12560
Policyholder Benefits Ceded             11304
Policyholder Benefits Gross             11304
Net Policyholder Benefits And Claims    11304
                                        ...  
Basic Average Shares                     1256
Operating Expense                        1256
Operating Income                         1256
Diluted EPS                              1256
state                                    1256
Length: 64, dtype: int64

In [10]:
# Columns to fill with 0
fill_zero_cols = [
    'Salaries And Wages', 'Earnings From Equity Interest', 
    'Loss Adjustment Expense', 'Policyholder Benefits Ceded', 
    'Policyholder Benefits Gross', 'Net Policyholder Benefits And Claims',
    'Write Off', 'Gain On Sale Of Ppe', 'Average Dilution Earnings',
    'Other Special Charges', 'Selling And Marketing Expense', 
    'Other Operating Expenses', 'Impairment Of Capital Assets',
    'Gain On Sale Of Business', 'Restructuring And Mergern Acquisition'
]

for col in fill_zero_cols:
    if col in master_df.columns:
        master_df[col] = master_df[col].fillna(0)

# Columns to fill with median
fill_median_cols = [
    'fiveYearAvgDividendYield', 'earningsQuarterlyGrowth', 'lastDividendValue',
    'lastDividendDate', 'debtToEquity', 'earningsGrowth', 'dividendDate',
    'Normalized EBITDA', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill',
    'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA',
    'Interest Income', 'Total Operating Income As Reported', 'Diluted Average Shares',
    'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Minority Interests',
    'Net Income Discontinuous Operations', 'Earnings From Equity Interest Net Of Tax',
    'Other Non Operating Income Expenses', 'Special Income Charges',
    'Interest Income Non Operating', 'Operating Income', 'Operating Expense',
    'Research And Development', 'Selling General And Administration',
    'General And Administrative Expense', 'Other Gand A',
    'Depreciation Amortization Depletion Income Statement',
    'Depreciation And Amortization In Income Statement',
    'Amortization', 'Amortization Of Intangibles Income Statement', 'Gross Profit',
    'Cost Of Revenue'
]

for col in fill_median_cols:
    if col in master_df.columns:
        median_value = master_df[col].median()
        master_df[col] = master_df[col].fillna(median_value)

# Check missing values per column in master_df
missing_counts_after = master_df.isna().sum()

# Optionally, show only columns that still have missing data (should be none)
missing_counts_after = missing_counts_after[missing_counts_after > 0]

print("Columns with missing data after cleaning:")
print(missing_counts_after)

Columns with missing data after cleaning:
state                                   1256
irWebsite                               1256
dividendRate                            2512
dividendYield                           2512
lastSplitFactor                         2512
lastSplitDate                           2512
trailingPegRatio                        2512
displayName                             3768
address2                                8792
fax                                     8792
Gain On Sale Of Security                6280
Otherunder Preferred Stock Dividend    10048
dtype: int64


In [11]:
# 1️⃣ Select the columns to include in features_df
feature_cols = [
    'Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume',
    'Dividends', 'Stock Splits', 'previousClose',
    'fiftyDayAverage', 'twoHundredDayAverage', 'fiftyTwoWeekLow', 'fiftyTwoWeekHigh',
    'trailingPE', 'forwardPE', 'priceToBook', 'priceToSalesTrailing12Months',
    'enterpriseValue', 'marketCap',
    'totalRevenue', 'Total Expenses', 'grossProfits', 'operatingCashflow', 'ebitda', 'EBITDA', 'EBIT', 'Net Income',
    'Average Dilution Earnings', 'trailingEps', 'forwardEps',
    'totalCash', 'totalDebt', 'currentRatio', 'quickRatio', 'debtToEquity', 'bookValue',
    'grossMargins', 'ebitdaMargins', 'operatingMargins', 'returnOnAssets', 'returnOnEquity', 'profitMargins',
    'earningsGrowth', 'revenueGrowth', 'beta'
]


# 2️⃣ Create a copy from the master_df
features_df = master_df[feature_cols].copy()

# 3️⃣ Ensure Date column is datetime type
features_df['Date'] = pd.to_datetime(features_df['Date'], errors='coerce', utc=True)

features_df

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,previousClose,...,bookValue,grossMargins,ebitdaMargins,operatingMargins,returnOnAssets,returnOnEquity,profitMargins,earningsGrowth,revenueGrowth,beta
0,MMM,2021-03-26 04:00:00+00:00,134.725150,136.079322,134.166742,136.030457,3202410,0.0,0.0,148.05,...,8.867,0.39915,0.24114,0.12408,0.07584,0.75501,0.13027,-0.199,0.02,1.131
1,MMM,2021-03-29 04:00:00+00:00,135.150983,137.314844,135.074195,136.630783,3111753,0.0,0.0,148.05,...,8.867,0.39915,0.24114,0.12408,0.07584,0.75501,0.13027,-0.199,0.02,1.131
2,MMM,2021-03-30 04:00:00+00:00,136.100250,137.251993,135.332433,135.862930,2244533,0.0,0.0,148.05,...,8.867,0.39915,0.24114,0.12408,0.07584,0.75501,0.13027,-0.199,0.02,1.131
3,MMM,2021-03-31 04:00:00+00:00,135.374320,135.716350,134.055052,134.494812,2902572,0.0,0.0,148.05,...,8.867,0.39915,0.24114,0.12408,0.07584,0.75501,0.13027,-0.199,0.02,1.131
4,MMM,2021-04-01 04:00:00+00:00,134.892676,135.360359,133.210448,134.508774,2274912,0.0,0.0,148.05,...,8.867,0.39915,0.24114,0.12408,0.07584,0.75501,0.13027,-0.199,0.02,1.131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12555,A,2026-03-20 04:00:00+00:00,111.570000,112.529999,110.180000,111.300003,4402500,0.0,0.0,112.98,...,24.436,0.52300,0.27898,0.22859,0.08533,0.19946,0.18259,-0.036,0.07,1.312
12556,A,2026-03-23 04:00:00+00:00,113.419998,113.699997,111.529999,112.019997,1975900,0.0,0.0,112.98,...,24.436,0.52300,0.27898,0.22859,0.08533,0.19946,0.18259,-0.036,0.07,1.312
12557,A,2026-03-24 04:00:00+00:00,110.589996,115.239998,110.029999,114.199997,2076900,0.0,0.0,112.98,...,24.436,0.52300,0.27898,0.22859,0.08533,0.19946,0.18259,-0.036,0.07,1.312
12558,A,2026-03-25 04:00:00+00:00,115.889999,116.400002,111.739998,112.980003,1704500,0.0,0.0,112.98,...,24.436,0.52300,0.27898,0.22859,0.08533,0.19946,0.18259,-0.036,0.07,1.312


In [16]:
# Ensure sorted
features_df = features_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# -------------------------------
# 1. RETURNS
# -------------------------------

# Daily return
features_df['return_1d'] = features_df.groupby('Ticker')['Close'].pct_change()

# Multi-day returns
features_df['return_5d'] = features_df.groupby('Ticker')['Close'].pct_change(5)
features_df['return_10d'] = features_df.groupby('Ticker')['Close'].pct_change(10)
features_df['return_21d'] = features_df.groupby('Ticker')['Close'].pct_change(21)

# -------------------------------
# 2. MOVING AVERAGES
# -------------------------------

features_df['ma_5'] = features_df.groupby('Ticker')['Close'].rolling(5).mean().reset_index(level=0, drop=True)
features_df['ma_10'] = features_df.groupby('Ticker')['Close'].rolling(10).mean().reset_index(level=0, drop=True)
features_df['ma_21'] = features_df.groupby('Ticker')['Close'].rolling(21).mean().reset_index(level=0, drop=True)

# -------------------------------
# 3. PRICE RELATIVE TO MOVING AVERAGES (Momentum)
# -------------------------------

features_df['price_ma5_ratio'] = features_df['Close'] / features_df['ma_5']
features_df['price_ma10_ratio'] = features_df['Close'] / features_df['ma_10']
features_df['price_ma21_ratio'] = features_df['Close'] / features_df['ma_21']

# -------------------------------
# 4. VOLATILITY (Rolling Std of Returns)
# -------------------------------

features_df['vol_5'] = features_df.groupby('Ticker')['return_1d'].rolling(5).std().reset_index(level=0, drop=True)
features_df['vol_10'] = features_df.groupby('Ticker')['return_1d'].rolling(10).std().reset_index(level=0, drop=True)
features_df['vol_21'] = features_df.groupby('Ticker')['return_1d'].rolling(21).std().reset_index(level=0, drop=True)

# -------------------------------
# 5. PRICE RANGE FEATURES
# -------------------------------

features_df['daily_range'] = features_df['High'] - features_df['Low']
features_df['range_pct'] = (features_df['High'] - features_df['Low']) / features_df['Close']

# -------------------------------
# 6. MOMENTUM (Trend Strength)
# -------------------------------

features_df['momentum_5'] = features_df['Close'] - features_df['Close'].shift(5)
features_df['momentum_10'] = features_df['Close'] - features_df['Close'].shift(10)
features_df['momentum_21'] = features_df['Close'] - features_df['Close'].shift(21)

features_df['momentum_5_pct'] = features_df['Close'].pct_change(5)
features_df['momentum_10_pct'] = features_df['Close'].pct_change(10)
features_df['momentum_21_pct'] = features_df['Close'].pct_change(21)


# -------------------------------
# 7. 52-WEEK POSITION
# -------------------------------

features_df['52w_position'] = (
    (features_df['Close'] - features_df['fiftyTwoWeekLow']) /
    (features_df['fiftyTwoWeekHigh'] - features_df['fiftyTwoWeekLow'])
)

# -------------------------------
# 8. GAP (Open vs Previous Close)
# -------------------------------

features_df['gap'] = (features_df['Open'] - features_df['previousClose']) / features_df['previousClose']

# -------------------------------
# 9. Risk Adjusted Return
# -------------------------------

features_df['risk_adjusted_return'] = features_df['return_5d'] / features_df['vol_5']

# -------------------------------
# 10. CLEAN NaNs (after feature creation)
# -------------------------------

features_df = features_df.fillna(0)
features_df


,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,previousClose,...,range_pct,momentum_5,momentum_10,momentum_21,52w_position,gap,momentum_5_pct,momentum_10_pct,momentum_21_pct,risk_adjusted_return
0,A,2021-03-26 04:00:00+00:00,118.428915,121.492563,117.936023,121.424911,1441900,0.0,0.0,112.98,...,0.029290,0.000000,0.000000,0.000000,0.391524,0.048229,0.000000,0.000000,0.000000,0.000000
1,A,2021-03-29 04:00:00+00:00,120.690388,121.598854,119.588635,121.212273,1539700,0.0,0.0,112.98,...,0.016584,0.000000,0.000000,0.000000,0.388193,0.068246,0.000000,0.000000,0.000000,0.000000
2,A,2021-03-30 04:00:00+00:00,120.429461,121.482889,120.236174,120.651741,1033700,0.0,0.0,112.98,...,0.010333,0.000000,0.000000,0.000000,0.379413,0.065936,0.000000,0.000000,0.000000,0.000000
3,A,2021-03-31 04:00:00+00:00,121.724501,124.169621,121.724501,122.874580,1815500,0.0,0.0,112.98,...,0.019899,0.000000,0.000000,0.000000,0.414232,0.077399,0.000000,0.000000,0.000000,0.000000
4,A,2021-04-01 04:00:00+00:00,123.705741,123.995679,122.748956,123.406143,1126400,0.0,0.0,112.98,...,0.010103,0.000000,0.000000,0.000000,0.422559,0.094935,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12555,MMM,2026-03-20 04:00:00+00:00,141.610001,143.300003,139.339996,141.199997,6900600,0.0,0.0,148.05,...,0.028045,-9.760010,-12.210007,-23.850006,0.346744,-0.043499,-0.064653,-0.079591,-0.144502,-7.540043
12556,MMM,2026-03-23 04:00:00+00:00,144.039993,148.220001,143.179993,146.559998,4924500,0.0,0.0,148.05,...,0.034389,-3.389999,-5.070007,-20.500000,0.443442,-0.027085,-0.022608,-0.033437,-0.122710,-0.908911
12557,MMM,2026-03-24 04:00:00+00:00,145.029999,147.800003,144.899994,146.669998,3036500,0.0,0.0,148.05,...,0.019772,-2.389999,-8.580002,-19.669998,0.445427,-0.020399,-0.016034,-0.055266,-0.118252,-0.642788
12558,MMM,2026-03-25 04:00:00+00:00,148.479996,149.169998,146.470001,148.050003,2238500,0.0,0.0,148.05,...,0.018237,2.970001,-7.119995,-18.410004,0.470323,0.002904,0.020471,-0.045885,-0.110597,0.959987


In [18]:
# -------------------------------
# 11. VOLUME-BASED FEATURES
# -------------------------------

# Rolling average volume (trend in trading activity)
features_df['volume_avg_5'] = features_df.groupby('Ticker')['Volume'].rolling(5).mean().reset_index(level=0, drop=True)
features_df['volume_avg_10'] = features_df.groupby('Ticker')['Volume'].rolling(10).mean().reset_index(level=0, drop=True)
features_df['volume_avg_21'] = features_df.groupby('Ticker')['Volume'].rolling(21).mean().reset_index(level=0, drop=True)

# Rolling volume volatility
features_df['volume_vol_5'] = features_df.groupby('Ticker')['Volume'].rolling(5).std().reset_index(level=0, drop=True)
features_df['volume_vol_10'] = features_df.groupby('Ticker')['Volume'].rolling(10).std().reset_index(level=0, drop=True)
features_df['volume_vol_21'] = features_df.groupby('Ticker')['Volume'].rolling(21).std().reset_index(level=0, drop=True)

# Volume momentum (change in trading activity)
features_df['volume_momentum_5'] = features_df['Volume'] - features_df.groupby('Ticker')['Volume'].shift(5)
features_df['volume_momentum_10'] = features_df['Volume'] - features_df.groupby('Ticker')['Volume'].shift(10)
features_df['volume_momentum_21'] = features_df['Volume'] - features_df.groupby('Ticker')['Volume'].shift(21)

# Volume momentum percentage
features_df['volume_momentum_5_pct'] = features_df['Volume'].pct_change(5)
features_df['volume_momentum_10_pct'] = features_df['Volume'].pct_change(10)
features_df['volume_momentum_21_pct'] = features_df['Volume'].pct_change(21)


# -------------------------------
# 12. TREND / MOMENTUM INDICATORS
# -------------------------------

# Rolling moving averages of close (already done: ma_5/10/21)
# We can add EMA (exponential moving average) for faster trend detection
features_df['ema_5'] = features_df.groupby('Ticker')['Close'].transform(lambda x: x.ewm(span=5, adjust=False).mean())
features_df['ema_10'] = features_df.groupby('Ticker')['Close'].transform(lambda x: x.ewm(span=10, adjust=False).mean())
features_df['ema_21'] = features_df.groupby('Ticker')['Close'].transform(lambda x: x.ewm(span=21, adjust=False).mean())

# MACD (12- and 26-day EMAs) and signal line
features_df['ema_12'] = features_df.groupby('Ticker')['Close'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
features_df['ema_26'] = features_df.groupby('Ticker')['Close'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
features_df['macd'] = features_df['ema_12'] - features_df['ema_26']
features_df['macd_signal'] = features_df.groupby('Ticker')['macd'].transform(lambda x: x.ewm(span=9, adjust=False).mean())

# RSI (Relative Strength Index)
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / (avg_loss + 1e-9)  # avoid division by 0
    rsi = 100 - (100 / (1 + rs))
    return rsi

features_df['rsi_14'] = features_df.groupby('Ticker')['Close'].transform(lambda x: compute_rsi(x, 14))


# -------------------------------
# 13. PRICE / VOLUME RATIOS & RELATIVES
# -------------------------------

# Price relative to EMA (trend confirmation)
features_df['close_ema5_ratio'] = features_df['Close'] / features_df['ema_5']
features_df['close_ema10_ratio'] = features_df['Close'] / features_df['ema_10']
features_df['close_ema21_ratio'] = features_df['Close'] / features_df['ema_21']

# Volume/price relative features
features_df['vol_price_ratio'] = features_df['Volume'] / (features_df['Close'] + 1e-9)
features_df['vol_ma5_ratio'] = features_df['Volume'] / (features_df['volume_avg_5'] + 1e-9)
features_df['vol_ma10_ratio'] = features_df['Volume'] / (features_df['volume_avg_10'] + 1e-9)
features_df['vol_ma21_ratio'] = features_df['Volume'] / (features_df['volume_avg_21'] + 1e-9)


# -------------------------------
# 14. CLEAN NaNs
# -------------------------------
features_df = features_df.fillna(0)
features_df

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,previousClose,...,macd,macd_signal,rsi_14,close_ema5_ratio,close_ema10_ratio,close_ema21_ratio,vol_price_ratio,vol_ma5_ratio,vol_ma10_ratio,vol_ma21_ratio
0,A,2021-03-26 04:00:00+00:00,118.428915,121.492563,117.936023,121.424911,1441900,0.0,0.0,112.98,...,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,11874.828503,0.000000,0.000000,0.000000
1,A,2021-03-29 04:00:00+00:00,120.690388,121.598854,119.588635,121.212273,1539700,0.0,0.0,112.98,...,-0.016963,-0.003393,0.000000,0.998832,0.998567,0.998408,12702.509130,0.000000,0.000000,0.000000
2,A,2021-03-30 04:00:00+00:00,120.429461,121.482889,120.236174,120.651741,1033700,0.0,0.0,112.98,...,-0.074774,-0.017669,0.000000,0.996134,0.995044,0.994352,8567.634343,0.000000,0.000000,0.000000
3,A,2021-03-31 04:00:00+00:00,121.724501,124.169621,121.724501,122.874580,1815500,0.0,0.0,112.98,...,0.058105,-0.002514,0.000000,1.009611,1.010917,1.011506,14775.228484,0.000000,0.000000,0.000000
4,A,2021-04-01 04:00:00+00:00,123.705741,123.995679,122.748956,123.406143,1126400,0.0,0.0,112.98,...,0.203954,0.038780,0.000000,1.009276,1.012476,1.014418,9127.584502,0.809521,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12555,MMM,2026-03-20 04:00:00+00:00,141.610001,143.300003,139.339996,141.199997,6900600,0.0,0.0,148.05,...,-5.247467,-3.721250,21.892335,0.974086,0.950637,0.919471,48871.105872,1.383829,1.371780,1.589081
12556,MMM,2026-03-23 04:00:00+00:00,144.039993,148.220001,143.179993,146.559998,4924500,0.0,0.0,148.05,...,-5.110159,-3.999032,34.487248,1.007348,0.989111,0.958349,33600.573704,0.994736,1.005135,1.102209
12557,MMM,2026-03-24 04:00:00+00:00,145.029999,147.800003,144.899994,146.669998,3036500,0.0,0.0,148.05,...,-4.935570,-4.186340,30.315610,1.005388,0.991683,0.962651,20702.938828,0.649460,0.658123,0.678805
12558,MMM,2026-03-25 04:00:00+00:00,148.479996,149.169998,146.470001,148.050003,2238500,0.0,0.0,148.05,...,-4.632453,-4.275562,37.568553,1.009850,1.000829,0.974214,15119.891617,0.523601,0.499373,0.501642
